# F1 Tyre Stint Strategy Explorer

Explore modern tyre compounds, completed-lap stint lengths, tyre age, and strategy sequences. Stint data begins in 2023; it does not contain lap times, so stint length is not a tyre-degradation measurement.


In [ ]:
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path(os.getenv("F1_DATA_DIR", "/kaggle/input/formula-1-pit-stop-dataset"))
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")
print(f"Reading data from {DATA_DIR}")


In [ ]:
stints = pd.read_csv(DATA_DIR / "stints.csv")
context = pd.read_csv(DATA_DIR / "race_context.csv")
stints["stint_length_laps"] = stints["lap_end"] - stints["lap_start"] + 1
valid = stints[stints["stint_length_laps"].gt(0)].copy()
valid["compound"] = valid["compound"].fillna("UNKNOWN").str.upper()
valid.head()


## Compound usage by season


In [ ]:
usage = valid.groupby(["season", "compound"]).size().rename("stints").reset_index()
usage["share"] = usage["stints"] / usage.groupby("season")["stints"].transform("sum")
pivot = usage.pivot(index="season", columns="compound", values="share").fillna(0)
pivot.plot.bar(stacked=True, figsize=(11, 5), colormap="tab20")
plt.title("Share of recorded stints by compound")
plt.ylabel("Share")
plt.legend(title="Compound", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()


## Stint length distributions


In [ ]:
common = valid[valid["compound"].isin(["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"])]
plt.figure(figsize=(11, 5))
sns.boxplot(data=common, x="compound", y="stint_length_laps", showfliers=False,
            order=["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"])
plt.title("Completed-lap stint lengths by compound")
plt.xlabel("")
plt.ylabel("Laps")
plt.tight_layout()
common.groupby("compound")["stint_length_laps"].agg(["count", "median", "mean"]).round(1)


## Strategy sequences

A sequence summarizes the ordered compounds for one driver-race. It is useful for exploration but does not encode safety-car timing, traffic, or tyre condition.


In [ ]:
strategy = (valid.sort_values(["season", "round_number", "driver_id", "stint_number"])
    .groupby(["season", "round_number", "driver_id"])["compound"]
    .agg(" → ".join).rename("strategy").reset_index())
strategy["stops_implied"] = strategy["strategy"].str.count("→")
display(strategy["strategy"].value_counts().head(15).to_frame("driver_races"))
plt.figure(figsize=(9, 5))
top = strategy["strategy"].value_counts().head(10).sort_values()
top.plot.barh(color="#e10600")
plt.title("Most common recorded compound sequences")
plt.xlabel("Driver-races")
plt.tight_layout()


## Build a race strategy table


In [ ]:
race_names = context[["season", "round_number", "circuit_short_name", "country_name"]]
strategy.merge(race_names, on=["season", "round_number"], how="left").sort_values(
    ["season", "round_number", "stops_implied"], ascending=[False, False, False]
).head(25)
